# Irodori-TTS Colab

Google Colab の GPU で Irodori-TTS の Gradio UI を起動するノートブックです。

Runtime > Change runtime type から `T4 GPU` などの GPU を選んでから、上から順番に実行してください。

今回追加した長文分割機能を使う場合は、この変更を含む自分の GitHub fork を `REPO_URL` に指定するか、変更済みのプロジェクト一式を Colab にアップロードして使ってください。

In [ ]:
# GPU 確認
!nvidia-smi

In [ ]:
# 設定
# 長文分割対応版を GitHub に置いた場合は、その URL に変更してください。
REPO_URL = "https://github.com/Aratako/Irodori-TTS.git"
BRANCH = "main"

# "normal" または "voicedesign"
APP_MODE = "normal"

# Hugging Face の非公開モデルを使う場合だけ入力してください。
HF_TOKEN = ""

# 既に /content/Irodori-TTS にアップロード済みのコードを使う場合は True
USE_EXISTING_REPO = False

In [ ]:
# リポジトリ取得
import os
from pathlib import Path

repo_dir = Path("/content/Irodori-TTS")

if USE_EXISTING_REPO:
    if not repo_dir.exists():
        raise FileNotFoundError("/content/Irodori-TTS がありません。アップロードするか USE_EXISTING_REPO=False にしてください。")
else:
    if repo_dir.exists():
        %cd /content/Irodori-TTS
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
        !git pull --ff-only origin {BRANCH}
    else:
        %cd /content
        !git clone --branch {BRANCH} {REPO_URL} Irodori-TTS

%cd /content/Irodori-TTS
!pwd
!git status --short || true

In [ ]:
# Colab / Gradio 互換パッチ
# 一部の Gradio では launch(css=...) が使えないため、Blocks 側に css を移します。
from pathlib import Path

for file in ["gradio_app.py", "gradio_app_voicedesign.py"]:
    path = Path(file)
    if not path.exists():
        print(f"skip missing file: {file}")
        continue
    text = path.read_text(encoding="utf-8")
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace("        css=EMOJI_PALETTE_CSS,\n", "")
    path.write_text(text, encoding="utf-8")

print("patched Gradio css compatibility")

In [ ]:
# 長文分割・結合パッチ
# 元の GitHub リポジトリを使う場合でも、Colab 上で長文対応コードを追加します。
from pathlib import Path

long_text_code = r'''
from __future__ import annotations

import re
from dataclasses import replace
from typing import Callable

import torch

from .inference_runtime import InferenceRuntime, SamplingRequest, SamplingResult


_TERMINAL_PUNCTUATION = "".join(chr(code) for code in (0x3002, 0xFF01, 0xFF1F))
_TERMINAL_PUNCTUATION += "!?。！？." + chr(0xFF0E)
_SOFT_BREAK_CHARS = "".join(chr(code) for code in (0x3001, 0xFF0C))
_SOFT_BREAK_CHARS += ",;" + chr(0xFF1B) + ":" + chr(0xFF1A) + " "
_SENTENCE_RE = re.compile(
    rf"[^{re.escape(_TERMINAL_PUNCTUATION)}\\n]+[{re.escape(_TERMINAL_PUNCTUATION)}]*|\\n+"
)


def split_text_for_tts(text: str, max_chars: int) -> list[str]:
    source = str(text).replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()
    if source == "":
        return []
    if max_chars <= 0 or len(source) <= max_chars:
        return [source]

    chunks: list[str] = []
    current = ""
    for match in _SENTENCE_RE.finditer(source):
        unit = match.group(0).strip()
        if unit == "":
            continue
        for piece in _split_oversized_unit(unit, max_chars):
            if current and len(current) + 1 + len(piece) > max_chars:
                chunks.append(current)
                current = piece
            elif current:
                current = _join_text_chunks(current, piece)
            else:
                current = piece

    if current:
        chunks.append(current)
    return chunks or [source]


def _join_text_chunks(left: str, right: str) -> str:
    if not left:
        return right
    if not right:
        return left
    if right[:1] in _TERMINAL_PUNCTUATION:
        return f"{left}{right}"
    if left[-1:].isascii() and left[-1:].isalnum() and right[:1].isascii() and right[:1].isalnum():
        return f"{left} {right}"
    return f"{left}{right}"


def _split_oversized_unit(unit: str, max_chars: int) -> list[str]:
    if len(unit) <= max_chars:
        return [unit]

    pieces: list[str] = []
    rest = unit.strip()
    while len(rest) > max_chars:
        window = rest[:max_chars]
        split_at = max(window.rfind(ch) for ch in _SOFT_BREAK_CHARS)
        if split_at < max_chars // 2:
            split_at = max_chars
        else:
            split_at += 1
        piece = rest[:split_at].strip()
        if piece:
            pieces.append(piece)
        rest = rest[split_at:].strip()
    if rest:
        pieces.append(rest)
    return pieces


def concatenate_audios(chunk_results: list[SamplingResult], *, silence_ms: int = 120) -> SamplingResult:
    if not chunk_results:
        raise ValueError("chunk_results must not be empty.")
    sample_rate = int(chunk_results[0].sample_rate)
    num_candidates = len(chunk_results[0].audios)
    silence_samples = max(0, int(sample_rate * max(0, int(silence_ms)) / 1000.0))

    merged: list[torch.Tensor] = []
    for candidate_idx in range(num_candidates):
        parts: list[torch.Tensor] = []
        for chunk_idx, result in enumerate(chunk_results):
            audio = result.audios[candidate_idx].float().cpu()
            parts.append(audio)
            if silence_samples > 0 and chunk_idx < len(chunk_results) - 1:
                parts.append(torch.zeros((audio.shape[0], silence_samples), dtype=audio.dtype))
        merged.append(torch.cat(parts, dim=-1))

    messages = [f"info: long text was split into {len(chunk_results)} chunks and concatenated."]
    for chunk_idx, result in enumerate(chunk_results, start=1):
        messages.append(f"chunk[{chunk_idx}] seed_used: {result.used_seed}")
        messages.extend(f"chunk[{chunk_idx}] {msg}" for msg in result.messages)

    stage_timings: list[tuple[str, float]] = []
    total_to_decode = 0.0
    for chunk_idx, result in enumerate(chunk_results, start=1):
        stage_timings.extend((f"chunk[{chunk_idx}].{name}", seconds) for name, seconds in result.stage_timings)
        total_to_decode += float(result.total_to_decode)

    return SamplingResult(
        audio=merged[0], audios=merged, sample_rate=sample_rate,
        stage_timings=stage_timings, total_to_decode=total_to_decode,
        used_seed=chunk_results[0].used_seed, messages=messages,
    )


def synthesize_long_text(runtime: InferenceRuntime, req: SamplingRequest, *, max_chars: int, silence_ms: int = 120, log_fn: Callable[[str], None] | None = None, log_prefix: str = "[long-text]") -> SamplingResult:
    chunks = split_text_for_tts(req.text, max_chars=max_chars)
    if len(chunks) <= 1:
        return runtime.synthesize(req, log_fn=log_fn)
    if log_fn is not None:
        log_fn(f"{log_prefix} split text into {len(chunks)} chunks (max_chars={max_chars})")

    base_seed = req.seed
    chunk_results: list[SamplingResult] = []
    for idx, chunk in enumerate(chunks, start=1):
        chunk_seed = None if base_seed is None else int(base_seed) + idx - 1
        if log_fn is not None:
            log_fn(f"{log_prefix} chunk {idx}/{len(chunks)} chars={len(chunk)}")
        chunk_results.append(runtime.synthesize(replace(req, text=chunk, seed=chunk_seed), log_fn=log_fn))

    result = concatenate_audios(chunk_results, silence_ms=silence_ms)
    if req.seconds is not None:
        result.messages.insert(1, "info: manual seconds is applied to each text chunk before concatenation.")
    return result
'''

Path("irodori_tts/long_text.py").write_text(long_text_code, encoding="utf-8")

def patch_app(path: Path, *, voice_design: bool) -> None:
    text = path.read_text(encoding="utf-8")
    if "from irodori_tts.long_text import synthesize_long_text" not in text:
        text = text.replace(
            "    save_wav,\n)\n",
            "    save_wav,\n)\nfrom irodori_tts.long_text import synthesize_long_text\n",
        )
    if "DEFAULT_LONG_TEXT_CHARS" not in text:
        text = text.replace(
            "GRADIO_AUDIO_COLS_PER_ROW = 8\n",
            "GRADIO_AUDIO_COLS_PER_ROW = 8\nDEFAULT_LONG_TEXT_CHARS = 140\nDEFAULT_CHUNK_SILENCE_MS = 180\n",
        )
    path.write_text(text, encoding="utf-8")

patch_app(Path("gradio_app.py"), voice_design=False)
patch_app(Path("gradio_app_voicedesign.py"), voice_design=True)

print("installed long-text helper. If the UI still has no Long Text controls, use a fork with the full patch or run the updated local notebook.")

In [ ]:
# Hugging Face ログイン。公開モデルだけ使う場合はスキップされます。
if HF_TOKEN.strip():
    !huggingface-cli login --token {HF_TOKEN}
else:
    print("HF_TOKEN is empty; skipping Hugging Face login.")

In [ ]:
# 依存関係インストール
# Colab の Python では sentencepiece<0.2 の wheel が無いことがあるため、Colab 用に差し替えます。
!python -m pip install -U pip setuptools wheel
!python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install sentencepiece==0.2.0
!grep -v '^sentencepiece' requirements.txt > requirements_colab.txt
!python -m pip install -r requirements_colab.txt

In [ ]:
# インストール確認
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Gradio UI 起動
# ログに表示される https://xxxxx.gradio.live を開いてください。
import shlex

if APP_MODE.lower().strip() == "voicedesign":
    app_file = "gradio_app_voicedesign.py"
    port = 7861
else:
    app_file = "gradio_app.py"
    port = 7860

cmd = f"python {shlex.quote(app_file)} --server-name 0.0.0.0 --server-port {port} --share"
print(cmd)
!{cmd}

## 長文分割の設定

長文分割対応版では、Gradio UI の `Advanced (Optional)` に次の項目が出ます。

- `Long Text Chunk Chars (0=off)`: 1 チャンクあたりの文字数。デフォルトは `180`。
- `Chunk Silence ms`: 結合時にチャンク間へ入れる無音。デフォルトは `120` ms。

`Seconds` を手動指定した場合は、各チャンクごとにその秒数が適用されてから結合されます。長文では `Seconds` を空欄にして自動推定を使うのがおすすめです。